# MIDA507 -- Session 04
## Metadonnees DCAT et Catalogage Documentaire Numerique

| | |
|---|---|
| **Programme** | Master MIDA -- Universite de Douala |
| **Session** | S04 -- Metadonnees DCAT et catalogage |
| **Duree estimee** | 90 minutes |
| **Prerequis** | Notebooks S01, S02, S03 |

### Ce que vous allez apprendre
1. Ce qu'est une metadonnee et pourquoi c'est votre metier au quotidien
2. La progression Dublin Core -> DCAT : du catalogue au portail open data
3. La structure DCAT en 3 niveaux (Catalogue / Dataset / Distribution)
4. Construire la fiche DCAT complete de notre jeu BU-UNG
5. Generer la notice HTML lisible par les usagers du portail

### Livrables : `MIDA507_S04_fiche_dcat.json` + `MIDA507_S04_notice.html`


In [ ]:
!pip install pandas seaborn matplotlib openpyxl --quiet
import pandas as pd, seaborn as sns, matplotlib.pyplot as plt
import json, os, hashlib, random
from datetime import datetime, timedelta
plt.rcParams['figure.figsize']=(12,5)
sns.set_theme(style='whitegrid')
print('Outils prets.')


In [ ]:
random.seed(42)
NB=5000
TYPES=["These et memoire","Manuel universitaire","Revue scientifique",
       "Rapport de recherche","Ouvrage de reference","Document administratif"]
FIL=["Droit","Sciences eco.","Lettres","Histoire","Geographie","Medecine","Agronomie","Informatique"]
NIV=["Licence 1","Licence 2","Licence 3","Master 1","Master 2","Doctorat"]
REG=["Adamaoua","Centre","Est","Extreme-Nord","Littoral","Nord","Ouest","Sud"]
LNG=["Francais","Anglais","Bilingue","Arabe","Autres"]
d0=datetime(2018,1,1)
emprunts=pd.DataFrame({
    "cote":[f"UNG-{str(i+1).zfill(5)}" for i in range(NB)],
    "type_doc":random.choices(TYPES,weights=[0.28,0.30,0.15,0.10,0.10,0.07],k=NB),
    "date":[(d0+timedelta(days=random.randint(0,365*5))).strftime("%Y-%m-%d") for _ in range(NB)],
    "duree":random.choices([3,7,7,14,14,14,21,30],k=NB),
    "usager":[f"USR{str(random.randint(1,800)).zfill(4)}" for _ in range(NB)],
    "filiere":random.choices(FIL,k=NB),
    "niveau":random.choices(NIV,k=NB),
    "region":random.choices(REG,k=NB),
    "langue":random.choices(LNG,weights=[0.62,0.22,0.10,0.04,0.02],k=NB),
})
emprunts["annee"]=pd.to_datetime(emprunts["date"]).dt.year
print(f"Jeu BU-UNG : {len(emprunts):,} emprunts, {len(emprunts.columns)} colonnes.")


---
## NOTION 1 -- Qu'est-ce qu'une Metadonnee ?

### Definition en 3 lignes

Une **metadonnee** est une information qui decrit une autre information. Elle ne contient pas les donnees elles-memes, mais les contextualise : qui les a produites, quand, pourquoi, sous quelle licence, dans quel format.

**Analogie IDA -- c'est votre metier :** La notice MARC d'un livre EST une metadonnee. La fiche d'archive d'un versement EST une metadonnee. L'en-tete d'un plan de classement EST une metadonnee. L'IDA produit des metadonnees depuis toujours.

**La nouveaute avec l'open data :** Les metadonnees doivent respecter le standard DCAT (W3C) pour que les moteurs de recherche et les portails africains puissent les lire automatiquement.

### Evolution des standards : du catalogue a l'open data

| Standard | Domaine | Ce qu'il decrit | Usage africain |
|---|---|---|---|
| **Dublin Core** | Bibliotheques (1995) | Documents textuels (15 champs) | Notices SIGB Koha, PMB |
| **MARC 21 / UNIMARC** | Bibliotheques (1966) | Notices bibliographiques detaillees | Catalogues universitaires |
| **EAD** | Archives (2002) | Instruments de recherche archivistiques | Archives nationales |
| **DCAT (W3C)** | Open Data (2014) | Jeux de donnees et leurs distributions | data.gouv.bj, opendata.sn |

> L'IDA qui maitrise DCAT peut travailler sur tout portail open data africain ou international.


In [ ]:
# NOTION 1 EN PRATIQUE -- Comparer Dublin Core et DCAT
print("COMPARAISON : Notice Dublin Core vs Fiche DCAT")
print("=" * 60)
print()
notice_dc = {
    "dc:title":       "Emprunts documentaires -- Bibliotheque UNG 2018-2023",
    "dc:creator":     "Bibliotheque Centrale, Universite de Ngaoundere",
    "dc:subject":     "Bibliotheque universitaire; emprunts; Cameroun",
    "dc:description": "5000 enregistrements d'emprunts sur 5 ans",
    "dc:date":        "2024",
    "dc:type":        "Dataset",
    "dc:format":      "text/csv",
    "dc:language":    "fra",
    "dc:rights":      "CC-BY 4.0",
}
fiche_dcat = {
    "dct:identifier":    "ark:/67375/UNG-BIBLIO-2018-2023",        # AJOUT : identifiant persistant
    "dct:title":         "Emprunts documentaires -- Bibliotheque UNG 2018-2023",
    "dct:creator":       "Bibliotheque Centrale, UNG",
    "dcat:keyword":      ["bibliotheque","Cameroun","emprunts","education"],  # AJOUT : liste de mots-cles
    "dct:description":   "5000 enregistrements d'emprunts sur 5 ans",
    "dct:issued":        "2024-01-15",              # AJOUT : date precise ISO 8601
    "dcat:mediaType":    "text/csv",
    "dct:language":      "fra",
    "dct:license":       "https://creativecommons.org/licenses/by/4.0/",  # AJOUT : URI complete
    "dcat:distribution": [{"url":"https://ckan.demo.ung.cm/data.csv"}],    # AJOUT : lien de telechargement
    "dct:publisher":     {"name":"BU-UNG","email":"data@univ-ngaoundere.cm"},  # AJOUT : contact
    "dct:spatial":       "Ngaoundere, Adamaoua, Cameroun",         # AJOUT : localisation
}
print("NOTICE DUBLIN CORE (ce que vous faites deja dans un SIGB) :")
for k,v in notice_dc.items():
    print(f"  {k:<20} : {v}")
print()
print("FICHE DCAT (evolution pour l'open data) :")
AJOUTS = {"dct:identifier","dcat:keyword","dct:issued","dct:license","dcat:distribution","dct:publisher","dct:spatial"}
for k,v in fiche_dcat.items():
    tag = " [NOUVEAU]" if k in AJOUTS else ""
    if isinstance(v,list): v = str(v[:2])
    if isinstance(v,dict): v = str(list(v.values())[:2])
    print(f"  {k:<25} : {str(v)[:50]}{tag}")
print()
print("APPORTS DCAT par rapport a Dublin Core :")
print("  - Identifiant ARK persistant --> garantit la trouvabilite longue duree")
print("  - URI de licence --> lisible automatiquement par les moteurs")
print("  - Lien de telechargement --> accessibilite directe")
print("  - Contact --> reutilisabilite (on sait qui contacter)")
print("  - Localisation --> contexte geographique")


---
## NOTION 2 -- Structure DCAT en 3 Niveaux

### Definition en 3 lignes

DCAT organise les informations sur **3 niveaux imbriques**, comme un plan de classement archivistique : le fonds (Catalogue), les series (Datasets), les pieces (Distributions).

**Analogie IDA -- plan de classement :**
- **Catalogue** = Fonds --> l'institution proprietaire, le portail
- **Dataset** = Serie --> un jeu de donnees specifique avec ses metadonnees
- **Distribution** = Piece --> le fichier CSV ou l'API concretes

### Pourquoi 3 niveaux et pas un seul ?

Parce qu'un meme jeu de donnees peut etre distribue sous plusieurs formes : le meme Dataset BU-UNG peut avoir une distribution CSV (pour les chercheurs), une distribution JSON (pour les developpeurs d'applications) et une distribution API REST (pour les ONG qui ont des applications mobiles). Les metadonnees DCAT s'adaptent a chaque forme sans se repeter.


In [ ]:
# NOTION 2 EN PRATIQUE -- Structure DCAT en 3 niveaux
print("STRUCTURE DCAT -- Bibliotheque Centrale UNG")
print("=" * 60)
print()
catalogue = {
    "type":          "dcat:Catalog",
    "dct:title":     "Portail Open Data -- Universite de Ngaoundere",
    "dct:publisher": "Direction des Systemes d'Information -- UNG",
    "dcat:dataset":  ["ung-biblio-emprunts-2018-2023"]
}
dataset = {
    "type":              "dcat:Dataset",
    "dct:identifier":    "ark:/67375/UNG-BIBLIO-2018-2023",
    "dct:title":         "Emprunts documentaires -- BU-UNG 2018-2023",
    "dct:description":   f"{len(emprunts):,} enregistrements d'emprunt, 10 colonnes",
    "dcat:keyword":      ["bibliotheque-universitaire","Cameroun","mida507"],
    "dct:license":       "https://creativecommons.org/licenses/by/4.0/",
    "dct:issued":        datetime.now().strftime("%Y-%m-%d"),
    "dcat:distribution": ["dist-csv","dist-api"]
}
distributions = [
    {"type":"dcat:Distribution","id":"dist-csv",
     "dct:title":"CSV UTF-8 -- Version complete",
     "dcat:downloadURL":"https://ckan.demo.ung.cm/data.csv",
     "dct:format":"CSV","dcat:mediaType":"text/csv"},
    {"type":"dcat:Distribution","id":"dist-api",
     "dct:title":"API REST JSON -- CKAN Datastore",
     "dcat:accessURL":"https://ckan.demo.ung.cm/api/3/action/datastore_search?resource_id=ung-emprunts",
     "dct:format":"API","dcat:mediaType":"application/json"},
]
print("NIVEAU 1 -- dcat:Catalog (le portail)")
for k,v in catalogue.items():
    print(f"  {k:<20} : {v}")
print()
print("NIVEAU 2 -- dcat:Dataset (le jeu de donnees)")
for k,v in dataset.items():
    if isinstance(v,list): v = v[:2]
    print(f"  {k:<20} : {str(v)[:55]}")
print()
print("NIVEAU 3 -- dcat:Distribution (fichiers concrets)")
for d in distributions:
    url = d.get("dcat:downloadURL") or d.get("dcat:accessURL","")
    print(f"  [{d['dct:format']}] {d['dct:title']}")
    print(f"       URL : {url}")
print()
print("AVANTAGE : un chercheur telecharge le CSV, une ONG utilise l'API,")
print("un journaliste consulte la notice HTML -- le meme Dataset sert tout le monde.")


---
## NOTION 3 -- Fiche DCAT Complete et Notice HTML

### En 3 lignes

La **fiche DCAT complete** est la notice numerique d'un jeu de donnees. Elle contient les **champs obligatoires** (sans lesquels le portail refuse le jeu) et les **champs recommandes** (qui ameliorent la trouvabilite).

La **notice HTML** est la page web que voit l'usager sur le portail. Elle doit etre lisible sur smartphone (81% des acces africains).


In [ ]:
# NOTION 3 EN PRATIQUE -- Generer la fiche DCAT complete
fiche = {
    "@context": "https://www.w3.org/ns/dcat#",
    "@type": "dcat:Dataset",
    # Champs OBLIGATOIRES
    "dct:identifier":  "ark:/67375/UNG-BIBLIO-EMPRUNTS-2018-2023",
    "dct:title":       {"fr": "Emprunts documentaires -- Bibliotheque Centrale UNG 2018-2023"},
    "dct:description": {"fr": (
        f"Jeu tabulaire de {len(emprunts):,} enregistrements d'emprunt documentaire "
        "de la Bibliotheque Centrale de l'Universite de Ngaoundere, couvrant 2018 a 2023. "
        f"{len(emprunts.columns)} colonnes : type de document, date, filiere, niveau, "
        "region d'origine, langue. Usagers pseudonymises (codes USR)."
    )},
    "dcat:keyword":    ["bibliotheque-universitaire","emprunt-documentaire","Cameroun",
                        "Adamaoua","sciences-information","education-superieure","MIDA507"],
    "dct:publisher":   {"foaf:name":"Bibliotheque Centrale -- UNG",
                        "foaf:mbox":"data@univ-ngaoundere.cm"},
    "dct:license":     "https://creativecommons.org/licenses/by/4.0/",
    "dcat:distribution": [
        {"dcat:downloadURL":"https://ckan.demo.ung.cm/dataset/ung-biblio/data.csv",
         "dcat:mediaType":"text/csv","dct:format":"CSV",
         "dct:description":f"{len(emprunts):,} lignes, {len(emprunts.columns)} colonnes, UTF-8"},
        {"dcat:accessURL":"https://ckan.demo.ung.cm/api/3/action/datastore_search?resource_id=ung-emprunts",
         "dcat:mediaType":"application/json","dct:format":"API REST"}
    ],
    # Champs RECOMMANDES
    "dct:issued":      datetime.now().strftime("%Y-%m-%d"),
    "dct:modified":    datetime.now().strftime("%Y-%m-%d"),
    "dct:language":    sorted(emprunts["langue"].dropna().unique().tolist()),
    "dct:spatial":     {"rdfs:label":"Ngaoundere, Region de l'Adamaoua, Cameroun","ISO_3166":"CM"},
    "dct:temporal":    {"dcat:startDate":emprunts["date"].min(),"dcat:endDate":emprunts["date"].max()},
    "dcat:theme":      ["http://publications.europa.eu/resource/authority/data-theme/EDUC"],
}
OBLIGATOIRES = ["dct:identifier","dct:title","dct:description","dcat:keyword",
                "dct:publisher","dct:license","dcat:distribution"]
RECOMMANDES  = ["dct:issued","dct:modified","dct:language","dct:spatial","dct:temporal","dcat:theme"]
print("BILAN COMPLETUDE DCAT-AP")
print("=" * 55)
print("  Champs obligatoires :")
for c in OBLIGATOIRES:
    val = fiche.get(c,"")
    if isinstance(val,dict): val = list(val.values())[0] if val else ""
    if isinstance(val,list): val = f"[{len(val)} elements]"
    print(f"    OK {c:<30} : {str(val)[:40]}")
print()
print("  Champs recommandes :")
for c in RECOMMANDES:
    present = c in fiche
    val = fiche.get(c,"")
    if isinstance(val,dict): val = str(list(val.values())[:1])[:35]
    if isinstance(val,list): val = f"[{len(val)} elements]"
    print(f"    {'OK' if present else 'ABSENT'} {c:<30} : {str(val)[:40]}")
score_ob  = sum(1 for c in OBLIGATOIRES if c in fiche)
score_rec = sum(1 for c in RECOMMANDES  if c in fiche)
print()
print(f"  Score DCAT : {score_ob}/{len(OBLIGATOIRES)} obligatoires + {score_rec}/{len(RECOMMANDES)} recommandes")
with open("MIDA507_S04_fiche_dcat.json","w",encoding="utf-8") as f:
    json.dump(fiche,f,ensure_ascii=False,indent=2)
print()
print("OK Fiche DCAT sauvegardee : MIDA507_S04_fiche_dcat.json")


In [ ]:
def generer_notice(fiche, df):
    titre = fiche["dct:title"]["fr"] if isinstance(fiche["dct:title"],dict) else fiche["dct:title"]
    desc  = fiche["dct:description"]["fr"] if isinstance(fiche["dct:description"],dict) else ""
    mots  = " ".join(
        '<span style="background:#BF360C;color:white;padding:2px 7px;border-radius:10px;'
        'font-size:11px;margin:2px;display:inline-block">' + m + '</span>'
        for m in fiche.get("dcat:keyword",[])[:7]
    )
    dists = ""
    for d in fiche.get("dcat:distribution",[]):
        fmt = d.get("dct:format","")
        url = d.get("dcat:downloadURL") or d.get("dcat:accessURL","")
        dists += (
            '<div style="border:1px solid #ddd;border-radius:6px;padding:10px;margin:6px 0;background:#f9f9f9">'
            '<strong style="color:#BF360C">' + fmt + '</strong>'
            '<a href="' + url + '" style="color:#1565C0;font-size:11px;display:block;margin-top:4px">' + url + '</a>'
            '</div>'
        )
    ark  = fiche.get("dct:identifier","")
    lic  = fiche.get("dct:license","")
    pub  = fiche.get("dct:publisher",{}).get("foaf:name","")
    date = fiche.get("dct:issued","")
    html = (
        '<!DOCTYPE html><html lang="fr"><head>'
        '<meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1">'
        '<title>' + titre + '</title>'
        '<style>'
        'body{font-family:Arial,sans-serif;max-width:780px;margin:0 auto;padding:16px;color:#333}'
        'h1{color:#3E1500;font-size:18px;border-bottom:3px solid #E64A19;padding-bottom:8px}'
        '.meta{background:#FBE9E7;border-radius:8px;padding:14px;margin:12px 0}'
        'table{width:100%;border-collapse:collapse;font-size:13px}'
        'td{padding:4px 8px;vertical-align:top}'
        '.stat{display:inline-block;background:#E3F2FD;border-radius:6px;padding:8px 12px;margin:4px;text-align:center;min-width:90px}'
        '.stat strong{display:block;font-size:18px;color:#1565C0}'
        'h2{color:#1565C0;font-size:15px;margin:18px 0 8px 0}'
        '</style></head><body>'
        '<h1>' + titre + '</h1>'
        '<div class="meta"><table>'
        '<tr><td style="font-weight:bold;color:#BF360C;width:160px">Identifiant</td>'
        '<td><code>' + ark + '</code></td></tr>'
        '<tr><td style="font-weight:bold;color:#BF360C">Editeur</td><td>' + pub + '</td></tr>'
        '<tr><td style="font-weight:bold;color:#BF360C">Date</td><td>' + date + '</td></tr>'
        '<tr><td style="font-weight:bold;color:#BF360C">Licence</td>'
        '<td><a href="' + lic + '" style="color:#1565C0">CC-BY 4.0</a></td></tr>'
        '<tr><td style="font-weight:bold;color:#BF360C">Mots-cles</td><td>' + mots + '</td></tr>'
        '</table></div>'
        '<h2>Description</h2>'
        '<p style="line-height:1.6;font-size:13px">' + desc + '</p>'
        '<h2>Statistiques</h2>'
        '<div>'
        '<div class="stat"><strong>' + str(len(df)) + '</strong>lignes</div>'
        '<div class="stat"><strong>' + str(len(df.columns)) + '</strong>colonnes</div>'
        '<div class="stat"><strong>2018-2023</strong>periode</div>'
        '</div>'
        '<h2>Telecharger</h2>' + dists +
        '<hr style="margin:18px 0"><p style="font-size:11px;color:#999">'
        'Fiche DCAT-AP -- MIDA507 -- Universite de Douala. ' + date + '.</p>'
        '</body></html>'
    )
    return html

html_notice = generer_notice(fiche, emprunts)
with open("MIDA507_S04_notice.html","w",encoding="utf-8") as f:
    f.write(html_notice)
print(f"Notice HTML sauvegardee : MIDA507_S04_notice.html ({os.path.getsize('MIDA507_S04_notice.html')} octets)")
print()
from IPython.display import display, HTML
display(HTML(html_notice))


---
## EXERCICE -- Redigez la fiche DCAT de votre institution


In [ ]:
# EXERCICE -- Fiche DCAT de mon institution
MON_TITRE   = "[A COMPLETER : Titre descriptif long de votre jeu]"
MON_ARK     = "[A COMPLETER : ark:/NAAN/institution-type-annee]"
MON_EDITEUR = "[A COMPLETER : Nom complet de votre institution]"
MA_DESC     = "[A COMPLETER : Description : sujet, periode, couverture, methode]"
MES_MOTS    = ["[tag1]","[tag2]","[tag3]"]
MA_LICENCE  = "https://creativecommons.org/licenses/by/4.0/"
MON_CSV_URL = "[A COMPLETER : URL du fichier CSV]"

remplis = sum(1 for v in [MON_TITRE,MON_ARK,MON_EDITEUR,MA_DESC] if "[A COMPLETER]" not in str(v))
print("VERIFICATION FICHE DCAT -- Mon institution")
print("=" * 50)
print(f"  Titre   : {MON_TITRE}")
print(f"  ARK     : {MON_ARK}")
print(f"  Editeur : {MON_EDITEUR}")
print(f"  Licence : {MA_LICENCE}")
print(f"  Tags    : {MES_MOTS}")
print()
if remplis == 4:
    ma_fiche = {
        "dct:identifier": MON_ARK,
        "dct:title":      {"fr": MON_TITRE},
        "dct:description":{"fr": MA_DESC},
        "dcat:keyword":   MES_MOTS,
        "dct:publisher":  {"foaf:name": MON_EDITEUR},
        "dct:license":    MA_LICENCE,
        "dcat:distribution": [{"dcat:downloadURL": MON_CSV_URL,"dct:format":"CSV"}],
        "dct:issued":     datetime.now().strftime("%Y-%m-%d"),
    }
    with open("MIDA507_S04_ma_fiche_dcat.json","w",encoding="utf-8") as f:
        json.dump(ma_fiche,f,ensure_ascii=False,indent=2)
    print("OK Fiche sauvegardee : MIDA507_S04_ma_fiche_dcat.json")
else:
    print("[A COMPLETER] Remplissez les champs entre guillemets.")
    print("Guide :")
    print("  MON_ARK = 'ark:/12345/ins-cameroun-emprunts-2023'")
    print("  MES_MOTS = ['cameroun','statistiques','enquete','2023','INS']")


---
## Bilan de la Session 04

| Competence | Acquise |
|---|---|
| Expliquer metadonnee et lien avec le metier IDA | OK |
| Situer DCAT dans la progression Dublin Core --> open data | OK |
| Comprendre la structure DCAT (3 niveaux) | OK |
| Generer une fiche DCAT complete (7 champs obligatoires) | OK |
| Produire la notice HTML responsive | OK |
| Rediger la fiche DCAT de mon institution | A completer |

### Prochaine session
**S05 -- Qualite des donnees :** auditer et nettoyer notre jeu selon les 6 dimensions DAMA.

*Notebook MIDA507 S04 -- Master MIDA -- Universite de Douala*
